# Verification Pipeline (IAA -> QC)

## 공통설정

In [1]:
!pip install -q openai google-genai
import json, time
from collections import Counter
from openai import OpenAI
from google import genai
from google.genai import types
from google.colab import userdata
from tqdm.auto import tqdm

oai = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
gem = genai.Client(api_key=userdata.get('GEMINI_API'))
O3_MODEL = 'o3'
GEMINI_MODEL = 'gemini-3.1-flash-lite'

In [2]:
# ============================================================
# [유틸] 공통 헬퍼 함수들
# ============================================================

def chunks(lst, n):
    """리스트를 n개씩 잘라 배치로 넘겨주는 제너레이터. (예: 50건 → 5건씩 10배치)"""
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def get_nested(d, path):
    """'output.label' 같은 중첩 경로로 값 꺼내기. CON은 'label'(최상위), PEP/RPT는 'output.label'(중첩)이라 필요."""
    cur = d
    for k in path.split('.'):
        cur = cur.get(k) if isinstance(cur, dict) else None
    return cur

def parse_arr(txt):
    """LLM 응답 텍스트(JSON)를 배열로 변환.
    - 최상위가 배열이면 그대로
    - {"results":[...]}처럼 객체로 감싸여 오면 그 안의 배열을 꺼냄
    - 그마저 없으면 객체 하나를 배열에 담아 반환"""
    if not txt: return []
    p = json.loads(txt)
    if isinstance(p, list): return p
    for v in p.values():
        if isinstance(v, list): return v
    return [p]


# ============================================================
# [지표] IAA 측정용 (순수 파이썬, sklearn 불필요)
# ============================================================

def cohen_kappa(g, pr, labels):
    """Cohen's kappa 계산. g=내 골든 라벨, pr=AI 판정.
    우연히 맞을 확률(pe)을 제거한 일치도. 1에 가까울수록 내 기준과 AI가 잘 맞음.
    반환: (kappa, 단순일치율 po, 비교한 건수 n)"""
    ids = [i for i in g if i in pr]; n = len(ids)
    if n == 0: return None, 0.0, 0
    po = sum(1 for i in ids if g[i]==pr[i]) / n          # 실제 일치 비율
    pe = 0.0
    for L in labels:
        pe += (sum(1 for i in ids if g[i]==L)/n) * (sum(1 for i in ids if pr[i]==L)/n)  # 우연 일치 기대치
    return ((po-pe)/(1-pe) if pe < 1 else 1.0), po, n

def interp(k):
    """kappa 값을 사람이 읽을 등급으로 변환 (0.8↑ 거의완벽 / 0.6↑ 상당 / 0.4↑ 보통 / 그 이하 낮음)."""
    if k is None: return 'N/A'
    return ('almost-perfect(>=0.8)' if k>0.8 else 'substantial(0.6-0.8)' if k>0.6
            else 'moderate(0.4-0.6)' if k>0.4 else 'low(<0.4)')

def confusion(g, pr, labels):
    """혼동행렬. 행=내 골든, 열=AI 판정. 어디서 엇갈리는지(예: 내가 검토인데 AI는 일치) 파악용."""
    m = {a:{b:0 for b in labels} for a in labels}
    for i in g:
        if i in pr and g[i] in labels and pr[i] in labels:
            m[g[i]][pr[i]] += 1
    return m


# ============================================================
# [IAA] 재판정 프롬프트 + 입력 구성
# ============================================================

def _instruction(labels, boundary, criteria_rules):
    """AI에게 줄 판정 지시문 생성.
    내 판정 기준(criteria_rules=RULES)을 넣어 '내 기준대로' 판정하게 하고,
    정답 label은 숨겨서(앵커링 방지) 독립 재판정시킴. 출력은 판정+이유만."""
    lab = '|'.join(labels)
    return (
        '너는 공공 SW 문서 검토자다. 각 건의 입력만 보고 아래 기준으로 판정하라.\n'
        f'- 판정값은 {labels} 중 하나다.\n'
        '- 근거는 주어진 입력 안에만 둔다. 외부 지식으로 없는 기준을 채우지 마라.\n'
        f'- {boundary}\n'
        f'{criteria_rules}\n'
        '- 입력에 정답 label은 없다. 위 기준으로 독립 판정한다.\n'
        f'출력은 JSON 배열만: [{{"id":"...","판정":"{lab}","이유":"한 문장 근거"}}]'
    )

def _payload(batch, input_fields):
    """레코드에서 AI에 줄 입력 필드만 뽑아 정리. (정답 label은 절대 안 넣음)
    CON은 [clause_text, refs], PEP/RPT는 [criteria, 발췌문들]."""
    out = []
    for it in batch:
        item = {'id': it['id']}
        for f in input_fields:
            item[f.split('.')[-1]] = get_nested(it, f) if '.' in f else it.get(f, '')
        out.append(item)
    return out


# ============================================================
# [IAA] o3 / Gemini 각각 재판정 호출
# ============================================================

def judge_o3(batch, instr, input_fields, retries=3):
    """o3(OpenAI)로 한 배치 재판정. 실패 시 최대 3회 재시도. 판정 배열 반환."""
    for a in range(retries):
        try:
            r = oai.chat.completions.create(
                model=O3_MODEL,
                messages=[{'role':'system','content':instr},
                          {'role':'user','content':json.dumps(_payload(batch,input_fields), ensure_ascii=False)}],
                response_format={'type':'json_object'}, max_completion_tokens=8000)
            txt = r.choices[0].message.content
            if not txt: time.sleep(3); continue   # 빈 응답이면 재시도
            return parse_arr(txt)
        except Exception as e:
            print('  [o3]', e); time.sleep(5)
    return []

def judge_gemini(batch, instr, input_fields, retries=3):
    """Gemini로 한 배치 재판정. o3와 동일 로직, SDK만 다름."""
    for a in range(retries):
        try:
            r = gem.models.generate_content(
                model=GEMINI_MODEL,
                contents=instr + '\n\n입력:\n' + json.dumps(_payload(batch,input_fields), ensure_ascii=False),
                config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
            if not r.text: time.sleep(3); continue
            return parse_arr(r.text)
        except Exception as e:
            print('  [gemini]', e); time.sleep(5)
    return []

def run_model(judge_fn, name, data, instr, input_fields, labels, batch):
    """한 모델(o3 or Gemini)로 전체 데이터를 배치로 돌려 판정+이유 수집.
    반환: (pred={id:판정}, reasons={id:이유})"""
    pred = {}
    reasons = {}
    for b in tqdm(list(chunks(data, batch)), desc=name, unit='batch'):
        for v in judge_fn(b, instr, input_fields):
            if 'id' in v and v.get('판정') in labels:
                pred[v['id']] = v['판정']
                reasons[v['id']] = v.get('이유', '')   # AI가 왜 그렇게 판정했는지
        time.sleep(2)
    return pred, reasons


# ============================================================
# [IAA 메인] o3 vs Gemini 비교 → 승자(검수기) 선정
# ============================================================

def iaa_compare(data, gold_field, labels, input_fields, boundary, criteria_rules, task):
    """확정 골든을 o3·Gemini 둘 다 재판정 → 각각 내 골든과 kappa 비교 → 승자 반환.
    - 각 모델의 accuracy/kappa/혼동행렬 출력
    - mismatch(내 골든≠AI) 건은 이유까지 화면에 찍음 (골든 재점검용)
    - 전체 판정+이유를 {task}_{모델}_all.json으로 저장
    - kappa 높은 모델 = 내 기준과 더 잘 맞는 검수기 → QC에 사용"""
    gold = {r['id']: (get_nested(r, gold_field) if '.' in gold_field else r.get(gold_field)) for r in data}
    instr = _instruction(labels, boundary, criteria_rules)
    p_o3, rz_o3 = run_model(judge_o3, f'{task} o3', data, instr, input_fields, labels, 5)
    p_gm, rz_gm = run_model(judge_gemini, f'{task} Gemini', data, instr, input_fields, labels, 10)
    for nm, pr, rz in [('o3', p_o3, rz_o3), ('Gemini', p_gm, rz_gm)]:
        k, po, n = cohen_kappa(gold, pr, labels)
        print(f'\n== [{task}] IAA my-gold vs {nm} (n={n}) ==')
        print(f'  accuracy={po:.3f} kappa={k:.3f} -> {interp(k)}')
        m = confusion(gold, pr, labels)
        print('        ' + ''.join(f'{l:>6}' for l in labels))
        for g in labels:
            print(f'  {g:>4} ' + ''.join(f'{m[g][p]:>6}' for p in labels))
        diff = [(i, gold[i], pr[i]) for i in gold if i in pr and gold[i]!=pr[i]]
        print(f'  mismatch {len(diff)}:')
        for i, g, p in diff:
            print(f'    {i}: my={g} / {nm}={p}  ← 이유: {rz.get(i,"")}')
        json.dump({i:{'판정':pr[i],'이유':rz.get(i,'')} for i in pr},
                  open(f'{task}_{nm}_all.json','w'), ensure_ascii=False, indent=2)
    k_o3,_,_ = cohen_kappa(gold, p_o3, labels)
    k_gm,_,_ = cohen_kappa(gold, p_gm, labels)
    winner = 'o3' if (k_o3 or 0) >= (k_gm or 0) else 'Gemini'   # kappa 높은 쪽 승
    print(f'\n>>> [{task}] IAA winner: {winner}  (o3={k_o3:.3f} / Gemini={k_gm:.3f}) -> QC uses {winner}')
    return winner


# ============================================================
# [QC] 증강본 검수 → accept/review/drop 3분류
# ============================================================

def run_qc(aug, qc_prompt, winner, task, batch=10):
    """IAA 이긴 모델로 증강본을 JUDGE 프롬프트로 검수.
    각 증강 케이스를 3버킷으로 분류:
    - drop: 명백 결함(근거진위 거짓/금지어/disclaimer 누락/형식오류/방향성 오류) → 자동 탈락
    - review: 라벨 이견(AI 재판정 ≠ 기재 label) → 사람이 확인
    - accept: 전부 통과 → 학습 데이터로 채택"""
    def qc_batch(b, retries=3):
        """한 배치를 승자 모델(o3 or Gemini)로 QC 호출."""
        msg = json.dumps(b, ensure_ascii=False)
        for a in range(retries):
            try:
                if winner == 'o3':
                    r = oai.chat.completions.create(
                        model=O3_MODEL,
                        messages=[{'role':'system','content':qc_prompt},
                                  {'role':'user','content':'[입력]\n'+msg}],
                        response_format={'type':'json_object'}, max_completion_tokens=12000)
                    txt = r.choices[0].message.content
                else:
                    r = gem.models.generate_content(
                        model=GEMINI_MODEL,
                        contents=qc_prompt + '\n\n[입력]\n' + msg,
                        config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
                    txt = r.text
                if not txt: time.sleep(3); continue
                return parse_arr(txt)
            except Exception as e:
                print('  [qc]', e); time.sleep(5)
        return []

    # 전체 증강본 검수 결과 수집
    verdicts = {}
    for b in tqdm(list(chunks(aug, batch)), desc=f'{task} QC({winner})', unit='batch'):
        for v in qc_batch(b):
            if 'id' in v: verdicts[v['id']] = v
        time.sleep(3)

    # 3분류 (CON/PEP/RPT 검수기 출력 필드명이 달라 .get 기본값으로 공통 방어)
    by = {r['id']: r for r in aug}; acc, rev, drop = [], [], []
    for rid, v in verdicts.items():
        rec = by.get(rid)
        if rec is None: rev.append({'id': rid, '문제점': ['id not found']}); continue
        문제 = v.get('문제점', []); 통과 = v.get('통과', False)
        라벨OK = v.get('라벨일치', v.get('판정일치', True)) and v.get('label일치', True)
        결함 = (v.get('근거진위', True) is False) or bool(v.get('금지어', [])) or (v.get('disclaimer', True) is False) \
               or (v.get('형식정상', True) is False) or (v.get('사유진위', True) is False) or (v.get('방향성정상', True) is False)
        if 결함: drop.append({'id': rid, '사유': 문제 or ['format/basis defect']})
        elif not 라벨OK: rev.append({'id': rid, '재판정': v.get('재판정', v.get('재계산label','')), '문제점': 문제})
        elif 통과: acc.append(rec)
        else: rev.append({'id': rid, '문제점': 문제 or ['not passed']})
    print(f'[{task}] accept {len(acc)} / review {len(rev)} / drop {len(drop)}')
    json.dump(acc, open(f'{task}_aug_accept.json','w'), ensure_ascii=False, indent=2)
    json.dump(rev, open(f'{task}_aug_review.json','w'), ensure_ascii=False, indent=2)
    json.dump(drop, open(f'{task}_aug_drop.json','w'), ensure_ascii=False, indent=2)
    print(f'-> {task}_aug_accept/review/drop.json saved')

In [3]:
CON_RULES = """[판정 기준]
- 일치: clause_text의 핵심 수치·조건이 refs.chunk_text 기준과 동일하고, refs에 비교 가능한 기준이 실제로 존재한다.
- 일치경계(B): 핵심 수치·조건은 refs 기준과 같고, refs가 세부조건·기간·예외·기산 방식·운영 절차까지는 명시하지 않아 일부 더 엄격해 보여도 단정하기 어려운 경우 — 판정은 '일치'.
- 불일치: 핵심 수치·조건이 refs 기준과 다르다(초과·미달·배제·면제·생략·의무 가중·권리 제한·순서 변경·조건 삭제).
- 검토(RAG miss): 쟁점을 판단할 decisive 기준이 refs에 없다. refs가 같은 주제로 보여도 핵심 수치·조건을 직접 비교할 기준이 없으면 검토. 외부 지식으로 정답 조문을 상정하지 않는다."""

PEP_RULES = """[특성별 판정 기준]
- 완전성: RFP 요구와 PEP 대응이 같은 산출물·같은 담당주체·같은 시점으로 읽히면 충족 / 요구 항목이 유사성조차 없거나 조건부 문구면 불가 / 동일 개념인지 애매하면 검토.
- 정확성: 수치·규격 정확 일치 충족 / 수치 불일치·PEP 내부 자기모순·비교 기준값 부재 불가 / 검토 없음.
- 검증가능성: 숫자·기한·비율·건수·도구명+기준 있으면 충족(블랙리스트 있어도 제거 후 독립검증 가능하면 충족) / 검증 기준 전무하면 불가 / 블랙리스트('적절히','최선을 다해','원활히 협의하여','필요한 범위에서','우수한 수준으로','성실히','무리 없이','신속히') 있고 유추 애매하면 검토.
- 추적성: RFP 과업·단계·산출물·일정·목표가 PEP에서 1:1 대응 충족 / 대응 항목 없음 불가 / 명칭 달라 애매하면 검토.
[종합] 불가 1개 이상이면 불가 / 불가 없고 검토 1개 이상이면 검토 / 전부 충족이면 충족."""

RPT_RULES = """[특성별 판정 기준]
- 완전성: PEP 과업·범위·산출물이 RPT에 모두 대응(동의어 인정, 같은 산출물·주체·시점 명백)하면 충족 / 하나라도 유사성조차 없거나 조건부('필요 시','~할 수 있다','추후 검토')면 불가 / 같은 산출물·주체·시점인지 갈리면 검토.
- 정확성: PEP 계획값과 RPT 결과값 직접 대조 일치 충족 / PEP값과 불일치·RPT 내부 자기모순·비교 기준값 부재 불가 / 검토 없음.
- 검증가능성: 수치·건수·기준일·비율·시험결과·검수결과·측정값으로 확인가능 충족(블랙리스트 있어도 제거 후 독립검증 가능하면 충족) / 검증 서술 없거나 블랙리스트만 있으면 불가 / 블랙리스트('적절히','최선을 다해','원활히 협의하여','필요한 범위에서','우수한 수준으로','성실히','무리 없이','신속히') 있고 유추 애매하면 검토.
- 추적성: PEP 단계·산출물과 RPT 결과 1:1 명확 대응 충족(보조신호 없어도 명확하면 충족) / 대응 안 되고 보조신호 3개(산출물명겹침·순번개수대응·기능설명일치) 모두 없으면 불가 / 재구성돼 보조신호 일부만 있어 1:1 확정 어려우면 검토.
[종합] 불가 1개 이상이면 불가 / 불가 없고 검토 1개 이상이면 검토 / 전부 충족이면 충족."""

## 계약서

In [ ]:
raw = json.load(open('con_seed_v3.json', encoding='utf-8'))
con_data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
con_winner = iaa_compare(
    con_data, 'label', ['일치','불일치','검토'],
    ['clause_text','refs'],
    "refs에 비교 기준 없으면 '검토'(RAG miss).",
    CON_RULES, 'CON')

CON o3:   0%|          | 0/10 [00:00<?, ?batch/s]

CON Gemini:   0%|          | 0/5 [00:00<?, ?batch/s]


== [CON] IAA my-gold vs o3 (n=35) ==
  accuracy=0.943 kappa=0.911 -> almost-perfect(>=0.8)
            일치   불일치    검토
    일치     10     1     0
   불일치      1    15     0
    검토      0     0     8
  mismatch 2:
    CON_017: my=일치 / o3=불일치  ← 이유: SW진흥법 제60조의 1년 책임기간을 명시하지 않아 핵심 조건(기간)을 생략하였다.
    CON_045: my=불일치 / o3=일치  ← 이유: 조문은 사업금액의 50% 이하 범위에서 사전승인을 받아 하도급하도록 규정하며, refs 소프트웨어진흥법 제51조①·⑤와 동일한 핵심 한도와 승인 요건을 갖춘다.

== [CON] IAA my-gold vs Gemini (n=50) ==
  accuracy=0.760 kappa=0.629 -> substantial(0.6-0.8)
            일치   불일치    검토
    일치     11     0     6
   불일치      1    20     2
    검토      0     3     7
  mismatch 12:
    CON_016: my=일치 / Gemini=검토  ← 이유: 하자보수보증금률은 공종별로 상이하게 규정되어 있어, 해당 계약의 성격(공종)을 알 수 없는 상태에서는 100분의 2가 적정한지 판단할 수 없습니다.
    CON_017: my=일치 / Gemini=검토  ← 이유: 하자보수보증금률은 공종별로 상이하게 규정되어 있어, 해당 계약의 성격(공종)을 알 수 없는 상태에서는 100분의 2가 적정한지 판단할 수 없습니다.
    CON_018: my=불일치 / Gemini=검토  ← 이유: 하자보수보증금률은 공종별로 상이하게 규정되어 있어, 해당 계약의 성격(공종)을 알 수 없는 상태에서는 100분의 5가 적정한지 판단할 수 없습니다.
   

In [6]:
con_winner = 'o3'
raw = json.load(open('CON_AUG_ONLY.json', encoding='utf-8'))
con_aug = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
CON_JUDGE_PROMPT = """당신은 공공 SW 계약서 검토 학습데이터의 품질 검수기다.

입력으로 증강 케이스 여러 건(JSON 배열)을 받으면, 각 건을 refs.chunk_text만으로 재판정하고 형식과 근거의 진위를 검사한다.
출력은 JSON 배열만 허용한다.

[중요 원칙]

* 판단은 반드시 각 건의 clause_text와 refs.chunk_text에만 근거한다.
* 외부 법 지식으로 보완하지 않는다.
* refs에 없는 기준을 "원래 그런 조문이 있다"고 가정해 통과시키지 않는다.
* refs.chunk_text 자체에 포함된 표현은 금지어 검사 대상이 아니다.
* 금지어 검사는 사고과정과 근거 본문에 대해서만 수행한다.

[재판정 절차]

1. clause_text에서 판정에 실질적으로 영향을 주는 핵심 수치·조건을 추출한다.
2. refs 중 실제 비교 기준을 제공하는 항목이 있는지 찾는다.
3. 기준이 있으면 clause_text와 refs.chunk_text를 직접 대조한다.
4. 기준이 없으면 "검토"로 본다.
5. pattern은 참고용 메타데이터일 뿐, 재판정은 clause_text와 refs로만 한다.
6. 단, pattern이 "일치경계(B)"인 경우에는 핵심 기준이 같고 세부 운영조건만 refs에 미명시된 구조인지 추가 확인한다.

[라벨 판정 기준]

1. 일치

* 재판정 결과는 "일치"다.
* clause_text의 핵심 수치·조건이 refs.chunk_text 기준과 동일하다.
* refs에 비교 기준이 실제로 존재해야 한다.

2. 일치경계(B)

* 최종 재판정은 여전히 "일치"다.
* 핵심 수치·조건은 refs 기준과 같다.
* 다만 세부조건, 기간, 예외, 기산 방식, 절차 등은 refs만으로 완전히 확정되지 않을 수 있다.
* clause_text가 다소 엄격해 보여도, 핵심 기준이 같고 세부조건이 refs에 명시되지 않으면 불일치로 바꾸지 않는다.
* 특히 refs가 원칙과 함께 예외·단서(재난 시 단축, 경제위기 시 완화 등)를 규정하는데 clause_text가 그 예외를 생략하고 원칙만 규정한 경우, 원칙 수치가 refs와 같으면 "일치"로 본다. 예외·세부 생략만을 이유로 "불일치"로 바꾸지 마라.

3. 불일치

* 재판정 결과는 "불일치"다.
* clause_text의 핵심 수치·조건이 refs 기준과 다르다.
* 차이 유형은 초과, 미달, 배제, 면제, 생략, 의무 가중, 권리 제한, 지급·검사 순서 변경 등일 수 있다.
* 단, refs의 예외·단서를 clause가 생략한 것은 불일치가 아니다. 핵심 수치·조건의 '값 자체'가 refs와 어긋날 때만 불일치다.

4. 검토(RAG miss)

* 재판정 결과는 "검토"다.
* clause_text 쟁점을 판정하는 decisive 기준이 refs.chunk_text에 없다.
* refs가 같은 주제처럼 보이더라도, 핵심 수치·조건을 직접 대조할 기준이 없으면 검토다.
* 외부 지식으로 보완해 일치/불일치로 판정하면 안 된다.

[검사 항목 — 각 건마다]

1. 라벨일치

* 재판정 결과가 기재된 label과 같은가
* label은 반드시 "일치", "불일치", "검토" 중 하나여야 한다

2. 근거진위

* 사고과정과 근거에서 언급한 법령명, 조문명, 조/항/호, 수치, 기간, 비율, 조건이 refs metadata 또는 refs.chunk_text에 실제로 존재하는가
* refs에 없는 법령명, 수치, 기준을 근거처럼 쓰면 false다

3. refs밖판단금지

* 제공되지 않은 조문, 법령명, 수치, 일반 법 지식을 끌어오지 않았는가

4. 금지어

* 사고과정과 근거 본문에서 다음 표현을 찾는다
  "위반", "부당", "무효", "적법", "위법", "소지", "정당"
* 발견되면 금지어 배열에 기록한다
* refs.chunk_text, law_name, article, category, pattern 안의 표현은 금지어 검사 대상이 아니다

5. disclaimer

* label이 "일치" 또는 "불일치"이면 근거는 반드시
  "(참고용 검토이며 법적 판단 아님)"
  으로 끝나야 한다.
* label이 "검토"이면 근거는 반드시
  "(참고자료 부족에 따른 검토 보류이며 법적 판단 아님)"
  으로 끝나야 한다.

6. 방향성정상

* label이 "불일치"인 경우 근거가 사실 관찰로 차이의 효과를 설명하는지 본다.
* 을불리/을유리로 자연스럽게 설명되는 category면 그 방향 설명이 있으면 정상이다.
* 그 구분이 자연스럽지 않은 category면, 어느 당사자 또는 어떤 계약 효과가 달라지는지 사실로 설명해도 정상이다.
* 법률결론어로 효과를 설명하면 false다.
* label이 "일치" 또는 "검토"이면 true로 둔다.

7. 형식정상

* 필수 필드 존재 여부를 확인한다:
  id, category, pattern, clause_text, refs, label, 근거
* 사고과정 필드가 있으면 그것도 검사 대상에 포함한다.
* refs는 배열이어야 하며, 각 ref에는 chunk_id, article, law_name, chunk_text가 있어야 한다.

[출력 형식 — 반드시 아래 JSON 객체 하나로, results 배열에 입력 건수만큼 모두 담아라]
{
  "results": [
    {
      "id": "...",
      "통과": true,
      "재판정": "일치|불일치|검토",
      "라벨일치": true,
      "근거진위": true,
      "refs밖판단금지": true,
      "금지어": [],
      "disclaimer": true,
      "방향성정상": true,
      "형식정상": true,
      "문제점": [],
      "신뢰도": 0.95
    }
  ]
}
반드시 입력의 모든 건을 results 배열에 포함하라. 한 건만 반환하지 마라.

[통과 조건]
아래를 모두 만족할 때만 통과=true다.

* 라벨일치=true
* 근거진위=true
* refs밖판단금지=true
* 금지어=[]
* disclaimer=true
* 방향성정상=true
* 형식정상=true

[입력]
{증강 레코드 JSON 배열}
"""

run_qc(con_aug, CON_JUDGE_PROMPT, con_winner, 'CON', batch=5)

CON QC(o3):   0%|          | 0/10 [00:00<?, ?batch/s]

[CON] accept 41 / review 0 / drop 1
-> CON_aug_accept/review/drop.json saved


## 과업수행계획서

In [ ]:
raw = json.load(open('pep_seed_golden.json', encoding='utf-8'))
pep_data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
pep_winner = iaa_compare(
    pep_data, 'output.label', ['충족','불가','검토'],
    ['input.criteria','input.rfp_excerpt','input.pep_excerpt'],
    "criteria 특성만 보고 대조로 확정 안 되면 '검토'. 정확성은 검토 금지.",
    PEP_RULES, 'PEP')

In [ ]:
raw = json.load(open('PEP_AUGMENTED.json', encoding='utf-8'))
pep_aug = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
PEP_JUDGE_PROMPT = """당신은 공공 SW 과업수행계획서(PEP) 검토 학습데이터의 품질 검수기다.
입력은 증강 레코드 여러 건의 JSON 배열이다.
각 건을 rfp_excerpt와 pep_excerpt만으로 재판정하고, 형식과 정합성을 검사한다.
출력은 JSON 배열만 허용한다.

[공통 원칙]
- 반드시 해당 건의 rfp_excerpt와 pep_excerpt에만 근거한다.
- 외부 지식으로 요구사항, 수치, 기준, 통상 항목을 보완하지 않는다.
- 법조문 refs, disclaimer 같은 부가 필드는 검사 기준이 아니다.
- 핵심은 output.eval의 특성별 판정과 사유가 실제 발췌문과 맞는지 여부다.

[입력 eval 파싱 규칙]
- output.eval은 문자열 배열이다.
- 각 원소는 반드시 "특성: 판정 — 사유" 형식이어야 한다.
- 각 문자열에서 특성명, 판정값, 사유를 파싱하여 검사한다.
- 특성명은 input.criteria에 포함된 값만 허용한다.
- 정확성의 판정값으로 "검토"가 나오면 형식 오류이자 판정 오류로 본다.

[재판정 규칙]

1. 완전성
- 충족:
  RFP 요구 항목이 PEP에 대응하며, 같은 산출물 또는 과업, 같은 담당주체, 같은 시점으로 읽히는 경우
- 불가:
  1) 요구 항목이 PEP 어디에도 없음
  2) 조건부 표현으로만 제시됨
- 검토:
  표현 차이로 인해 같은 항목인지 애매함

2. 정확성
- 충족:
  RFP와 PEP의 수치, 기간, 비율, 건수, 규격이 일치함
- 불가:
  1) PEP 내부 자기모순
  2) RFP와 PEP 수치 불일치
  3) 비교 기준값 부재
- 정확성은 검토를 허용하지 않는다

3. 검증가능성
- 블랙리스트 표현 예:
  "적절히", "최선을 다해", "원활히 협의하여", "필요한 범위에서", "우수한 수준으로", "성실히", "무리 없이", "신속히"
- 충족:
  검증 가능한 기준이 있고, 블랙리스트 표현이 있더라도 제거 후 남는 문장만으로 독립 검증 가능함
- 불가:
  검증 가능한 기준이 전혀 없거나 항목이 비어 있음
- 검토:
  블랙리스트 표현이 있으며, 제거 후에도 독립 검증 가능한지 애매함

4. 추적성
- 충족:
  RFP의 과업, 단계, 산출물, 일정 또는 목표가 PEP에서 1:1로 대응됨
- 불가:
  대응되는 항목이 없음
- 검토:
  명칭이나 구성은 달라졌지만 같은 항목인지 애매함
  또는 현재 발췌만으로 대응 여부를 단정하기 어려움

[검사 항목]
각 건마다 아래를 검사하라.

1) 판정일치
- 위 재판정 결과가 output.eval에서 파싱한 각 특성 판정과 일치하는가

2) label일치
- output.label이 output.eval의 판정들로부터 계산한 label과 일치하는가
- 계산 규칙:
  하나라도 불가면 불가
  불가 없고 하나라도 검토면 검토
  전부 충족이면 충족

3) 사유진위
- output.eval의 사유가 실제 발췌문에 근거하는가
- 없는 누락, 없는 수치불일치, 없는 모순, 없는 단절을 지어내지 않았는가

4) 방향성정상
- 판정과 사유가 서로 모순되지 않는가
- 예: 판정은 불가인데 사유는 "모두 대응함"이면 false

5) criteria보존
- input.criteria에 있는 특성만 output.eval에 있는가
- 빠진 특성이나 추가된 특성이 없는가
- 정확성에 검토가 들어가 있지 않은가

6) 형식정상
- id, input.criteria, input.rfp_excerpt, input.pep_excerpt, output.label, output.eval이 존재하는가
- output.eval의 각 원소가 "특성: 판정 — 사유" 형식인가
- 판정값이 허용된 값만 쓰였는가

[출력 형식]
반드시 JSON 배열만 출력한다.
[
  {
    "id": "...",
    "통과": true,
    "특성별재판정": {
      "완전성": "충족|불가|검토",
      "정확성": "충족|불가",
      "검증가능성": "충족|불가|검토",
      "추적성": "충족|불가|검토"
    },
    "재계산label": "충족|불가|검토",
    "판정일치": true,
    "label일치": true,
    "사유진위": true,
    "방향성정상": true,
    "criteria보존": true,
    "형식정상": true,
    "문제점": [],
    "신뢰도": 0.0
  }
]

[통과 조건]
- 판정일치 = true
- label일치 = true
- 사유진위 = true
- 방향성정상 = true
- criteria보존 = true
- 형식정상 = true

[신뢰도]
0.0~1.0 사이 값으로 주며, 발췌문 근거가 직접적이고 명확할수록 높인다.

[입력]
{증강 레코드 JSON 배열}
"""
run_qc(pep_aug, PEP_JUDGE_PROMPT, pep_winner, 'PEP')

## 결과보고서

In [ ]:
raw = json.load(open('rpt_seed_golden.json', encoding='utf-8'))
rpt_data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
rpt_winner = iaa_compare(
    rpt_data, 'output.label', ['충족','불가','검토'],
    ['input.criteria','input.pep_excerpt','input.rpt_excerpt'],
    "criteria 특성만 보고 대조로 확정 안 되면 '검토'. 정확성은 검토 금지.",
    RPT_RULES, 'RPT')

In [ ]:
raw = json.load(open('RPT_AUGMENTED.json', encoding='utf-8'))
rpt_aug = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
RPT_JUDGE_PROMPT = """당신은 공공 SW '사업추진결과보고서(RPT)' 검토 학습데이터 품질 검수기다.
입력은 증강 케이스 여러 건의 JSON 배열이다.
각 건을 독립적으로 재판정하고, output.eval과 output.label이 발췌문 근거와 일치하는지 검사한다.
출력은 JSON 배열만 허용한다.

[중요 원칙]
- 판단은 반드시 각 건의 pep_excerpt와 rpt_excerpt에만 근거한다.
- 외부 지식으로 없는 과업, 수치, 산출물, 기준을 보완하지 않는다.
- 법조문, refs, disclaimer 유무는 검사 기준이 아니다.
- input.criteria에 포함된 특성만 재판정한다.
- 먼저 스스로 재판정한 뒤, 그 결과를 output.eval 및 output.label과 대조한다.
- 정확성은 검토를 허용하지 않는다.

[입력 eval 파싱 규칙]
- output.eval은 문자열 배열이다.
- 각 원소는 반드시 "특성: 판정 — 사유" 형식이어야 한다.
- 각 문자열에서 특성명, 판정값, 사유를 파싱하여 검사한다.
- 특성명은 input.criteria에 포함된 값만 허용한다.
- 정확성에 "검토"가 있으면 형식 오류이자 판정 오류다.

[특성별 재판정 규칙]

1. 완전성(COMP)
- PEP의 과업, 범위, 산출물 항목마다 RPT에 대응이 있는지 확인한다.
- 동일 표현뿐 아니라 명백한 동의어와 패러프레이즈도 인정한다.
- 같은 산출물, 같은 주체, 같은 시점을 가리키는 것이 명백하면 충족이다.
- 하나라도 유사성조차 없거나 조건부 문구로만 처리되면 불가이다.
- 표현은 다르지만 같은 산출물, 같은 주체, 같은 시점인지 판단이 갈리면 검토이다.

2. 정확성(ACC)
- PEP의 계획값, 기준값, 기간, 예산, 건수, 규격과 RPT의 결과값이 직접 대조되어 일치하면 충족이다.
- 다음 중 하나라도 해당하면 불가이다.
  1) PEP 기준값과 RPT 결과값이 다름
  2) RPT 내부 자기모순
  3) 비교 가능한 기준값 부재
- 정확성은 검토를 사용하지 않는다.
- output.eval에 정확성: 검토가 있으면 그 자체로 형식 결함이자 판정 불일치로 본다.

3. 검증가능성(VER)
- 블랙리스트 표현 예:
  "적절히", "최선을 다해", "원활히 협의하여", "필요한 범위에서", "우수한 수준으로", "성실히", "무리 없이", "신속히"
- 수치, 건수, 기준일, 비율, 시험결과, 검수결과, 측정값 등 검증 가능한 기준이 전혀 없으면 불가이다.
- 검증 가능한 기준이 있고 블랙리스트 표현이 없으면 충족이다.
- 블랙리스트 표현이 있으면 해당 표현을 제거한 뒤에도 독립 검증 가능한지 본다.
  - 독립 검증 가능하면 충족
  - 독립 검증 불가능하면 불가
  - 판단이 갈리면 검토

4. 추적성(TRACE)
- PEP의 단계, 과업, 산출물, 일정, 목표와 RPT 결과의 연결을 발췌문 안에서 확인한다.
- 명확한 1:1 대응이 확인되면 충족이다.
- 대응 항목이 없거나 연결이 끊기면 불가이다.
- 단계명, 산출물명, 구성 방식이 달라 같은 항목인지 확정하기 어렵거나 현재 발췌문만으로 연결 정보가 부족하면 검토이다.
- 아래 3신호는 보조 판단 기준으로 활용할 수 있다.
  ① 산출물명 겹침
  ② 순번·개수 대응
  ③ 기능 설명 일치
- 3신호는 참고용이다. 반드시 3개가 모두 있어야 하는 것은 아니다.

[label 재계산 규칙]
- 특성별 재판정 중 불가가 1개 이상이면 재계산label = "불가"
- 불가가 없고 검토가 1개 이상이면 재계산label = "검토"
- 모든 특성이 충족이면 재계산label = "충족"

[검사 항목]

1. 판정일치
- 스스로 재판정한 결과가 output.eval에서 파싱한 각 특성 판정과 같은가

2. label일치
- 재계산label이 output.label과 같은가

3. 사유진위
- output.eval의 사유가 실제 발췌문에 근거하는가
- 없는 대응, 없는 누락, 없는 수치불일치, 없는 자기모순, 없는 단절을 지어내지 않았는가

4. 방향성정상
- 판정과 사유가 서로 모순되지 않는가
- 예: 판정은 불가인데 사유는 "모든 항목이 대응된다"라고 쓰면 false

5. criteria보존
- input.criteria에 있는 특성만 output.eval에 존재하는가
- 빠진 특성이나 추가된 특성이 없는가
- 정확성에 검토가 들어가 있지 않은가

6. 형식정상
- 필수 필드가 모두 존재하는가
  - id
  - input.criteria
  - input.pep_excerpt
  - input.rpt_excerpt
  - output.label
  - output.eval
- output.eval의 각 원소가 "특성: 판정 — 사유" 형식인가
- 각 판정값이 허용된 값만 쓰였는가

[출력 형식 — JSON 배열만, 입력 순서와 id 유지]
[
  {
    "id": "...",
    "통과": true,
    "특성별재판정": {
      "완전성": "충족|불가|검토",
      "정확성": "충족|불가",
      "검증가능성": "충족|불가|검토",
      "추적성": "충족|불가|검토"
    },
    "재계산label": "충족|불가|검토",
    "판정일치": true,
    "label일치": true,
    "사유진위": true,
    "방향성정상": true,
    "criteria보존": true,
    "형식정상": true,
    "문제점": [],
    "신뢰도": 0.0
  }
]

[출력 제약]
- 특성별재판정에는 해당 건의 input.criteria에 있는 특성만 넣는다.
- 문제점에는 실제 불일치나 형식 결함 사유를 짧게 적는다.
- 신뢰도는 0.0~1.0 사이 값으로 주며, 발췌문 근거가 직접적이고 명확할수록 높인다.

[통과 조건]
아래가 모두 true일 때만 통과다.
- 판정일치
- label일치
- 사유진위
- 방향성정상
- criteria보존
- 형식정상

[입력]
{증강 레코드 JSON 배열}
"""
run_qc(rpt_aug, RPT_JUDGE_PROMPT, rpt_winner, 'RPT')